In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
# 1. Load the CSV you just generated
csv_path = 'training_labels.csv'
df = pd.read_csv(csv_path)

# 2. Randomly sample 12000 images to prevent the SVM from crashing your RAM
# Ensure we get a balanced mix of classes (-1, 0, 1) if possible
SAMPLE_SIZE = 12000
if len(df) > SAMPLE_SIZE:
    df = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print(f"Dataset loaded and sampled down to {len(df)} images.")
print("\nClass distribution in our sample:")
print(df['label'].value_counts())

Dataset loaded and sampled down to 12000 images.

Class distribution in our sample:
label
 0    8619
 1    2306
-1    1075
Name: count, dtype: int64


In [3]:
def preprocess_image(image_path):
    """Loads an image, crops it, converts to grayscale, resizes, and flattens it."""
    # 1. Load image
    img = cv2.imread(image_path)
    if img is None:
        return None
    
    # 2. Crop the image to the Region of Interest (ROI)
    # The image is 720x1280. We only care about the bottom 40% where the line is.
    h, w = img.shape[:2]
    roi = img[int(h * 0.6):h, :]
    
    # 3. Convert to Grayscale (SVMs prefer 1 channel over 3 BGR channels)
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    
    # 4. Resize to a very small footprint (e.g., 64x32) so the SVM can process it fast
    resized = cv2.resize(gray, (64, 32))
    
    # 5. Flatten into a 1D array (size: 64 * 32 = 2048 features)
    features = resized.flatten()
    return features

print("Preprocessing function defined.")

Preprocessing function defined.


In [ ]:
image_dir = 'extracted_images'

X = []
y = []

print("Starting feature extraction. This might take a minute...")

for index, row in df.iterrows():
    img_path = os.path.join(image_dir, row['filename'])
    label = row['label']
    
    features = preprocess_image(img_path)
    
    if features is not None:
        X.append(features)
        y.append(label)

X = np.array(X)
y = np.array(y)

print(f"Feature extraction complete!")
print(f"X shape (samples, features): {X.shape}")
print(f"y shape (labels): {y.shape}")

Starting feature extraction. This might take a minute...


In [ ]:
# Create a Pipeline: Standardize the features, then train the SVM
svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    # We add class_weight='balanced' to heavily penalize missing the rare left/right turns
    ("svm", SVC(kernel="rbf", C=1.0, gamma="scale", class_weight='balanced'))
])

print("\nTraining the SVM model...")
svm_pipeline.fit(X_train, y_train)
print("Training complete!")

In [ ]:
# Predict on the test set
y_pred = svm_pipeline.predict(X_test)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=["LEFT (-1)", "STRAIGHT (0)", "RIGHT (1)"]))

# Plot confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, 
    display_labels=["LEFT (-1)", "STRAIGHT (0)", "RIGHT (1)"],
    cmap=plt.cm.Blues
)
plt.title("SVM Confusion Matrix")
plt.show()

In [ ]:
model_filename = 'svm_model.pkl'
joblib.dump(svm_pipeline, model_filename)

print(f"SUCCESS! Model saved as '{model_filename}'")
print("You are now ready to upload your .ipynb file for the May 19 deadline!")